# 1. P-median 문제

### 1.0 baseline

In [9]:
from ortools.sat.python import cp_model

demand_cities = ["강남", "강북", "인천", "수원", "성남", "고양", "부천",
                 "안양"]
candidates = ["김포", "강동", "의왕", "파주", "하남", "안산", "남양주"]

# 각 수요 도시의 월간 수요량 (톤)
weights = [120, 90, 150, 110, 80, 100, 95, 70]

# 거리 행렬 d[i][j]: 수요 도시 i -> 후보 창고 j (km)
# 김포 강동 의왕 파주 하남 안산 남양주
distances = [
    [38, 15, 22, 55, 20, 40, 28], # 강남
    [25, 30, 40, 32, 35, 55, 25], # 강북
    [18, 52, 48, 45, 58, 28, 62], # 인천
    [40, 38, 15, 65, 42, 22, 50], # 수원
    [45, 18, 16, 58, 22, 32, 30], # 성남
    [20, 38, 50, 18, 42, 58, 38], # 고양
    [15, 38, 32, 35, 45, 25, 48], # 부천
    [32, 35, 12, 60, 40, 18, 45], # 안양
]

n = len(demand_cities) # 수요 도시 수
m = len(candidates) # 후보 창고 수
p = 3 # 선택할 창고 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 결정변수
x = [model.NewBoolVar(f"x_{candidates[j]}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{demand_cities[i]}_{candidates[j]}")
     for j in range(m)] for i in range(n)]

# 제약 (1): 정확히 p개 창고 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요 도시는 정확히 1개 창고에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 창고에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 목적함수: 가중 거리 합계 최소화
# CP-SAT는 정수형만 처리하므로 실수 거리를 그대로 정수로 사용 (이미 정수 데이터) )
obj_terms = []
for i in range(n):
    for j in range(m):
        obj_terms.append(weights[i] * distances[i][j] * y[i][j])

model.Minimize(sum(obj_terms))

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    print(f"\n 풀이 상태 : {tag}")
    print(f" 목적함수값: {int(solver.ObjectiveValue()):,} (톤ㆍkm/월)")

    # 선택된 창고
    selected = [j for j in range(m) if solver.Value(x[j]) ==1]
    print(f"\n > 선택된 창고 ({p}곳):")
    for j in selected:
        print(f"- 후보{j+1}: {candidates[j]}")

    # 할당 결과
    print(f"\n > 수요 도시별 할당 창고:")
    print(f"{'수요 도시': <8} {'수요량':>7}{'배정 창고':<10}{'거리':>6} {'기여 비용':>10}")
    print(f"{'-'*50}")
    total_cost = 0
    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) ==1)
        cost = weights[i] * distances[i][assigned]
        total_cost += cost
        print(f"{demand_cities[i]:<8} {weights[i]:>5}톤 {candidates[assigned]:<10}"
              f"{distances[i][assigned]:>4}km {cost:>8,}톤ㆍkm")
    print(f"{'-'*50}")
    print(f"{'합계':<8} {sum(weights):>5}톤 {'':>4} {total_cost:>8,}톤ㆍkm")

    # 창고별 서비스 권역 요약
    print(f"\n > 창고별 서비스 권역:")
    for j in selected:
        served = [demand_cities[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        load = sum(weights[i] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f"{candidates[j]}: {', '.join(served)} (총 {load}톤)")


 풀이 상태 : 최적해
 목적함수값: 13,945 (톤ㆍkm/월)

 > 선택된 창고 (3곳):
- 후보1: 김포
- 후보2: 강동
- 후보3: 의왕

 > 수요 도시별 할당 창고:
수요 도시        수요량배정 창고         거리      기여 비용
--------------------------------------------------
강남         120톤 강동          15km    1,800톤ㆍkm
강북          90톤 김포          25km    2,250톤ㆍkm
인천         150톤 김포          18km    2,700톤ㆍkm
수원         110톤 의왕          15km    1,650톤ㆍkm
성남          80톤 의왕          16km    1,280톤ㆍkm
고양         100톤 김포          20km    2,000톤ㆍkm
부천          95톤 김포          15km    1,425톤ㆍkm
안양          70톤 의왕          12km      840톤ㆍkm
--------------------------------------------------
합계         815톤        13,945톤ㆍkm

 > 창고별 서비스 권역:
김포: 강북, 인천, 고양, 부천 (총 435톤)
강동: 강남 (총 120톤)
의왕: 수원, 성남, 안양 (총 260톤)


### 1.1

In [11]:
from ortools.sat.python import cp_model

demand_cities = ["강남", "강북", "인천", "수원", "성남", "고양", "부천",
                 "안양"]
candidates = ["김포", "강동", "의왕", "파주", "하남", "안산", "남양주"]

# 각 수요 도시의 월간 수요량 (톤)
weights = [120, 90, 150, 110, 80, 100, 95, 70]

# 거리 행렬 d[i][j]: 수요 도시 i -> 후보 창고 j (km)
# 김포 강동 의왕 파주 하남 안산 남양주
distances = [
    [38, 15, 22, 55, 20, 40, 28], # 강남
    [25, 30, 40, 32, 35, 55, 25], # 강북
    [18, 52, 48, 45, 58, 28, 62], # 인천
    [40, 38, 15, 65, 42, 22, 50], # 수원
    [45, 18, 16, 58, 22, 32, 30], # 성남
    [20, 38, 50, 18, 42, 58, 38], # 고양
    [15, 38, 32, 35, 45, 25, 48], # 부천
    [32, 35, 12, 60, 40, 18, 45], # 안양
]

n = len(demand_cities) # 수요 도시 수
m = len(candidates) # 후보 창고 수
p = 4 # 선택할 창고 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 결정변수
x = [model.NewBoolVar(f"x_{candidates[j]}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{demand_cities[i]}_{candidates[j]}")
     for j in range(m)] for i in range(n)]

# 제약 (1): 정확히 p개 창고 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요 도시는 정확히 1개 창고에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 창고에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 목적함수: 가중 거리 합계 최소화
# CP-SAT는 정수형만 처리하므로 실수 거리를 그대로 정수로 사용 (이미 정수 데이터) )
obj_terms = []
for i in range(n):
    for j in range(m):
        obj_terms.append(weights[i] * distances[i][j] * y[i][j])

model.Minimize(sum(obj_terms))

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    print(f"\n 풀이 상태 : {tag}")
    print(f" 목적함수값: {int(solver.ObjectiveValue()):,} (톤ㆍkm/월)")

    # 선택된 창고
    selected = [j for j in range(m) if solver.Value(x[j]) ==1]
    print(f"\n > 선택된 창고 ({p}곳):")
    for j in selected:
        print(f"- 후보{j+1}: {candidates[j]}")

    # 할당 결과
    print(f"\n > 수요 도시별 할당 창고:")
    print(f"{'수요 도시': <8} {'수요량':>7}{'배정 창고':<10}{'거리':>6} {'기여 비용':>10}")
    print(f"{'-'*50}")
    total_cost = 0
    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) ==1)
        cost = weights[i] * distances[i][assigned]
        total_cost += cost
        print(f"{demand_cities[i]:<8} {weights[i]:>5}톤 {candidates[assigned]:<10}"
              f"{distances[i][assigned]:>4}km {cost:>8,}톤ㆍkm")
    print(f"{'-'*50}")
    print(f"{'합계':<8} {sum(weights):>5}톤 {'':>4} {total_cost:>8,}톤ㆍkm")

    # 창고별 서비스 권역 요약
    print(f"\n > 창고별 서비스 권역:")
    for j in selected:
        served = [demand_cities[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        load = sum(weights[i] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f"{candidates[j]}: {', '.join(served)} (총 {load}톤)")


 풀이 상태 : 최적해
 목적함수값: 13,745 (톤ㆍkm/월)

 > 선택된 창고 (4곳):
- 후보1: 김포
- 후보2: 강동
- 후보3: 의왕
- 후보4: 파주

 > 수요 도시별 할당 창고:
수요 도시        수요량배정 창고         거리      기여 비용
--------------------------------------------------
강남         120톤 강동          15km    1,800톤ㆍkm
강북          90톤 김포          25km    2,250톤ㆍkm
인천         150톤 김포          18km    2,700톤ㆍkm
수원         110톤 의왕          15km    1,650톤ㆍkm
성남          80톤 의왕          16km    1,280톤ㆍkm
고양         100톤 파주          18km    1,800톤ㆍkm
부천          95톤 김포          15km    1,425톤ㆍkm
안양          70톤 의왕          12km      840톤ㆍkm
--------------------------------------------------
합계         815톤        13,745톤ㆍkm

 > 창고별 서비스 권역:
김포: 강북, 인천, 부천 (총 335톤)
강동: 강남 (총 120톤)
의왕: 수원, 성남, 안양 (총 260톤)
파주: 고양 (총 100톤)


1.2

In [12]:
from ortools.sat.python import cp_model

demand_cities = ["강남", "강북", "인천", "수원", "성남", "고양", "부천",
                 "안양"]
candidates = ["김포", "강동", "의왕", "파주", "하남", "안산", "남양주"]

# 각 수요 도시의 월간 수요량 (톤)
weights = [120, 90, 150, 110, 80, 100, 95, 70]

# 거리 행렬 d[i][j]: 수요 도시 i -> 후보 창고 j (km)
# 김포 강동 의왕 파주 하남 안산 남양주
distances = [
    [38, 15, 22, 55, 20, 40, 28], # 강남
    [25, 30, 40, 32, 35, 55, 25], # 강북
    [18, 52, 48, 45, 58, 28, 62], # 인천
    [40, 38, 15, 65, 42, 22, 50], # 수원
    [45, 18, 16, 58, 22, 32, 30], # 성남
    [20, 38, 50, 18, 42, 58, 38], # 고양
    [15, 38, 32, 35, 45, 25, 48], # 부천
    [32, 35, 12, 60, 40, 18, 45], # 안양
]

n = len(demand_cities) # 수요 도시 수
m = len(candidates) # 후보 창고 수
p = 3 # 선택할 창고 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 결정변수
x = [model.NewBoolVar(f"x_{candidates[j]}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{demand_cities[i]}_{candidates[j]}")
     for j in range(m)] for i in range(n)]

# 제약 (1): 정확히 p개 창고 선택
model.Add(sum(x) == p)

# 추가 제약: 김포(후보 0번)는 반드시 설치
model.Add(x[0] == 1)

# 제약 (2): 각 수요 도시는 정확히 1개 창고에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 창고에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 목적함수: 가중 거리 합계 최소화
# CP-SAT는 정수형만 처리하므로 실수 거리를 그대로 정수로 사용 (이미 정수 데이터) )
obj_terms = []
for i in range(n):
    for j in range(m):
        obj_terms.append(weights[i] * distances[i][j] * y[i][j])

model.Minimize(sum(obj_terms))

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    print(f"\n 풀이 상태 : {tag}")
    print(f" 목적함수값: {int(solver.ObjectiveValue()):,} (톤ㆍkm/월)")

    # 선택된 창고
    selected = [j for j in range(m) if solver.Value(x[j]) ==1]
    print(f"\n > 선택된 창고 ({p}곳):")
    for j in selected:
        print(f"- 후보{j+1}: {candidates[j]}")

    # 할당 결과
    print(f"\n > 수요 도시별 할당 창고:")
    print(f"{'수요 도시': <8} {'수요량':>7}{'배정 창고':<10}{'거리':>6} {'기여 비용':>10}")
    print(f"{'-'*50}")
    total_cost = 0
    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) ==1)
        cost = weights[i] * distances[i][assigned]
        total_cost += cost
        print(f"{demand_cities[i]:<8} {weights[i]:>5}톤 {candidates[assigned]:<10}"
              f"{distances[i][assigned]:>4}km {cost:>8,}톤ㆍkm")
    print(f"{'-'*50}")
    print(f"{'합계':<8} {sum(weights):>5}톤 {'':>4} {total_cost:>8,}톤ㆍkm")

    # 창고별 서비스 권역 요약
    print(f"\n > 창고별 서비스 권역:")
    for j in selected:
        served = [demand_cities[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        load = sum(weights[i] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f"{candidates[j]}: {', '.join(served)} (총 {load}톤)")


 풀이 상태 : 최적해
 목적함수값: 13,945 (톤ㆍkm/월)

 > 선택된 창고 (3곳):
- 후보1: 김포
- 후보2: 강동
- 후보3: 의왕

 > 수요 도시별 할당 창고:
수요 도시        수요량배정 창고         거리      기여 비용
--------------------------------------------------
강남         120톤 강동          15km    1,800톤ㆍkm
강북          90톤 김포          25km    2,250톤ㆍkm
인천         150톤 김포          18km    2,700톤ㆍkm
수원         110톤 의왕          15km    1,650톤ㆍkm
성남          80톤 의왕          16km    1,280톤ㆍkm
고양         100톤 김포          20km    2,000톤ㆍkm
부천          95톤 김포          15km    1,425톤ㆍkm
안양          70톤 의왕          12km      840톤ㆍkm
--------------------------------------------------
합계         815톤        13,945톤ㆍkm

 > 창고별 서비스 권역:
김포: 강북, 인천, 고양, 부천 (총 435톤)
강동: 강남 (총 120톤)
의왕: 수원, 성남, 안양 (총 260톤)


1.3

In [13]:
from ortools.sat.python import cp_model

demand_cities = ["강남", "강북", "인천", "수원", "성남", "고양", "부천",
                 "안양"]
candidates = ["김포", "강동", "의왕", "파주", "하남", "안산", "남양주"]

# 각 수요 도시의 월간 수요량 (톤)
weights = [120, 90, 150, 110, 80, 100, 95, 70]

# 거리 행렬 d[i][j]: 수요 도시 i -> 후보 창고 j (km)
# 김포 강동 의왕 파주 하남 안산 남양주
distances = [
    [38, 15, 22, 55, 20, 40, 28], # 강남
    [25, 30, 40, 32, 35, 55, 25], # 강북
    [18, 52, 48, 45, 58, 28, 62], # 인천
    [40, 38, 15, 65, 42, 22, 50], # 수원
    [45, 18, 16, 58, 22, 32, 30], # 성남
    [20, 38, 50, 18, 42, 58, 38], # 고양
    [15, 38, 32, 35, 45, 25, 48], # 부천
    [32, 35, 12, 60, 40, 18, 45], # 안양
]

n = len(demand_cities) # 수요 도시 수
m = len(candidates) # 후보 창고 수
p = 3 # 선택할 창고 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 결정변수
x = [model.NewBoolVar(f"x_{candidates[j]}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{demand_cities[i]}_{candidates[j]}")
     for j in range(m)] for i in range(n)]

# 제약 (1): 정확히 p개 창고 선택
model.Add(sum(x) == p)

# 추가 제약: 파주(후보 3번)는 반드시 설치x
model.Add(x[3] == 0)

# 제약 (2): 각 수요 도시는 정확히 1개 창고에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 창고에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 목적함수: 가중 거리 합계 최소화
# CP-SAT는 정수형만 처리하므로 실수 거리를 그대로 정수로 사용 (이미 정수 데이터) )
obj_terms = []
for i in range(n):
    for j in range(m):
        obj_terms.append(weights[i] * distances[i][j] * y[i][j])

model.Minimize(sum(obj_terms))

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    print(f"\n 풀이 상태 : {tag}")
    print(f" 목적함수값: {int(solver.ObjectiveValue()):,} (톤ㆍkm/월)")

    # 선택된 창고
    selected = [j for j in range(m) if solver.Value(x[j]) ==1]
    print(f"\n > 선택된 창고 ({p}곳):")
    for j in selected:
        print(f"- 후보{j+1}: {candidates[j]}")

    # 할당 결과
    print(f"\n > 수요 도시별 할당 창고:")
    print(f"{'수요 도시': <8} {'수요량':>7}{'배정 창고':<10}{'거리':>6} {'기여 비용':>10}")
    print(f"{'-'*50}")
    total_cost = 0
    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) ==1)
        cost = weights[i] * distances[i][assigned]
        total_cost += cost
        print(f"{demand_cities[i]:<8} {weights[i]:>5}톤 {candidates[assigned]:<10}"
              f"{distances[i][assigned]:>4}km {cost:>8,}톤ㆍkm")
    print(f"{'-'*50}")
    print(f"{'합계':<8} {sum(weights):>5}톤 {'':>4} {total_cost:>8,}톤ㆍkm")

    # 창고별 서비스 권역 요약
    print(f"\n > 창고별 서비스 권역:")
    for j in selected:
        served = [demand_cities[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        load = sum(weights[i] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f"{candidates[j]}: {', '.join(served)} (총 {load}톤)")


 풀이 상태 : 최적해
 목적함수값: 13,945 (톤ㆍkm/월)

 > 선택된 창고 (3곳):
- 후보1: 김포
- 후보2: 강동
- 후보3: 의왕

 > 수요 도시별 할당 창고:
수요 도시        수요량배정 창고         거리      기여 비용
--------------------------------------------------
강남         120톤 강동          15km    1,800톤ㆍkm
강북          90톤 김포          25km    2,250톤ㆍkm
인천         150톤 김포          18km    2,700톤ㆍkm
수원         110톤 의왕          15km    1,650톤ㆍkm
성남          80톤 의왕          16km    1,280톤ㆍkm
고양         100톤 김포          20km    2,000톤ㆍkm
부천          95톤 김포          15km    1,425톤ㆍkm
안양          70톤 의왕          12km      840톤ㆍkm
--------------------------------------------------
합계         815톤        13,945톤ㆍkm

 > 창고별 서비스 권역:
김포: 강북, 인천, 고양, 부천 (총 435톤)
강동: 강남 (총 120톤)
의왕: 수원, 성남, 안양 (총 260톤)


1.4

In [16]:
from ortools.sat.python import cp_model

demand_cities = ["강남", "강북", "인천", "수원", "성남", "고양", "부천",
                 "안양"]
candidates = ["김포", "강동", "의왕", "파주", "하남", "안산", "남양주"]

# 각 수요 도시의 월간 수요량 (톤)
weights = [120, 90, 150, 110, 80, 100, 95, 70]

# 거리 행렬 d[i][j]: 수요 도시 i -> 후보 창고 j (km)
# 김포 강동 의왕 파주 하남 안산 남양주
distances = [
    [38, 15, 22, 55, 20, 40, 28], # 강남
    [25, 30, 40, 32, 35, 55, 25], # 강북
    [18, 52, 48, 45, 58, 28, 62], # 인천
    [40, 38, 15, 65, 42, 22, 50], # 수원
    [45, 18, 16, 58, 22, 32, 30], # 성남
    [20, 38, 50, 18, 42, 58, 38], # 고양
    [15, 38, 32, 35, 45, 25, 48], # 부천
    [32, 35, 12, 60, 40, 18, 45], # 안양
]

n = len(demand_cities) # 수요 도시 수
m = len(candidates) # 후보 창고 수
p = 3 # 선택할 창고 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 결정변수
x = [model.NewBoolVar(f"x_{candidates[j]}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{demand_cities[i]}_{candidates[j]}")
     for j in range(m)] for i in range(n)]

# 제약 (1): 정확히 p개 창고 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요 도시는 정확히 1개 창고에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 창고에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 제약 (4): 각 창고의 월 처리용량은 최대 300톤
for j in range(m):
    model.Add(sum(weights[i] * y[i][j] for i in range(n)) <= 300*x[j])

# 목적함수: 가중 거리 합계 최소화
# CP-SAT는 정수형만 처리하므로 실수 거리를 그대로 정수로 사용 (이미 정수 데이터) )
obj_terms = []
for i in range(n):
    for j in range(m):
        obj_terms.append(weights[i] * distances[i][j] * y[i][j])

model.Minimize(sum(obj_terms))

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    print(f"\n 풀이 상태 : {tag}")
    print(f" 목적함수값: {int(solver.ObjectiveValue()):,} (톤ㆍkm/월)")

    # 선택된 창고
    selected = [j for j in range(m) if solver.Value(x[j]) ==1]
    print(f"\n > 선택된 창고 ({p}곳):")
    for j in selected:
        print(f"- 후보{j+1}: {candidates[j]}")

    # 할당 결과
    print(f"\n > 수요 도시별 할당 창고:")
    print(f"{'수요 도시': <8} {'수요량':>7}{'배정 창고':<10}{'거리':>6} {'기여 비용':>10}")
    print(f"{'-'*50}")
    total_cost = 0
    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) ==1)
        cost = weights[i] * distances[i][assigned]
        total_cost += cost
        print(f"{demand_cities[i]:<8} {weights[i]:>5}톤 {candidates[assigned]:<10}"
              f"{distances[i][assigned]:>4}km {cost:>8,}톤ㆍkm")
    print(f"{'-'*50}")
    print(f"{'합계':<8} {sum(weights):>5}톤 {'':>4} {total_cost:>8,}톤ㆍkm")

    # 창고별 서비스 권역 요약
    print(f"\n > 창고별 서비스 권역:")
    for j in selected:
        served = [demand_cities[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        load = sum(weights[i] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f"{candidates[j]}: {', '.join(served)} (총 {load}톤)")


 풀이 상태 : 최적해
 목적함수값: 16,170 (톤ㆍkm/월)

 > 선택된 창고 (3곳):
- 후보1: 김포
- 후보2: 강동
- 후보3: 의왕

 > 수요 도시별 할당 창고:
수요 도시        수요량배정 창고         거리      기여 비용
--------------------------------------------------
강남         120톤 강동          15km    1,800톤ㆍkm
강북          90톤 강동          30km    2,700톤ㆍkm
인천         150톤 김포          18km    2,700톤ㆍkm
수원         110톤 의왕          15km    1,650톤ㆍkm
성남          80톤 강동          18km    1,440톤ㆍkm
고양         100톤 김포          20km    2,000톤ㆍkm
부천          95톤 의왕          32km    3,040톤ㆍkm
안양          70톤 의왕          12km      840톤ㆍkm
--------------------------------------------------
합계         815톤        16,170톤ㆍkm

 > 창고별 서비스 권역:
김포: 인천, 고양 (총 250톤)
강동: 강남, 강북, 성남 (총 290톤)
의왕: 수원, 부천, 안양 (총 275톤)


1.5

In [15]:
from ortools.sat.python import cp_model

demand_cities = ["강남", "강북", "인천", "수원", "성남", "고양", "부천",
                 "안양"]
candidates = ["김포", "강동", "의왕", "파주", "하남", "안산", "남양주"]

# 각 수요 도시의 월간 수요량 (톤)
weights = [120, 90, 150, 110, 80, 100, 95, 70]

# 거리 행렬 d[i][j]: 수요 도시 i -> 후보 창고 j (km)
# 김포 강동 의왕 파주 하남 안산 남양주
distances = [
    [38, 15, 22, 55, 20, 40, 28], # 강남
    [25, 30, 40, 32, 35, 55, 25], # 강북
    [18, 52, 48, 45, 58, 28, 62], # 인천
    [40, 38, 15, 65, 42, 22, 50], # 수원
    [45, 18, 16, 58, 22, 32, 30], # 성남
    [20, 38, 50, 18, 42, 58, 38], # 고양
    [15, 38, 32, 35, 45, 25, 48], # 부천
    [32, 35, 12, 60, 40, 18, 45], # 안양
]

n = len(demand_cities) # 수요 도시 수
m = len(candidates) # 후보 창고 수
p = 3 # 선택할 창고 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 결정변수
x = [model.NewBoolVar(f"x_{candidates[j]}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{demand_cities[i]}_{candidates[j]}")
     for j in range(m)] for i in range(n)]

# 제약 (1): 정확히 p개 창고 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요 도시는 정확히 1개 창고에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 창고에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 제약 (4): 수요 도시 ~ 배정된 창고 거리 40km 초과 불가
for i in range(n):
    for j in range(m):
        if distances[i][j] > 40:
            model.Add(y[i][j]==0)

# 목적함수: 가중 거리 합계 최소화
# CP-SAT는 정수형만 처리하므로 실수 거리를 그대로 정수로 사용 (이미 정수 데이터) )
obj_terms = []
for i in range(n):
    for j in range(m):
        obj_terms.append(weights[i] * distances[i][j] * y[i][j])

model.Minimize(sum(obj_terms))

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    print(f"\n 풀이 상태 : {tag}")
    print(f" 목적함수값: {int(solver.ObjectiveValue()):,} (톤ㆍkm/월)")

    # 선택된 창고
    selected = [j for j in range(m) if solver.Value(x[j]) ==1]
    print(f"\n > 선택된 창고 ({p}곳):")
    for j in selected:
        print(f"- 후보{j+1}: {candidates[j]}")

    # 할당 결과
    print(f"\n > 수요 도시별 할당 창고:")
    print(f"{'수요 도시': <8} {'수요량':>7}{'배정 창고':<10}{'거리':>6} {'기여 비용':>10}")
    print(f"{'-'*50}")
    total_cost = 0
    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) ==1)
        cost = weights[i] * distances[i][assigned]
        total_cost += cost
        print(f"{demand_cities[i]:<8} {weights[i]:>5}톤 {candidates[assigned]:<10}"
              f"{distances[i][assigned]:>4}km {cost:>8,}톤ㆍkm")
    print(f"{'-'*50}")
    print(f"{'합계':<8} {sum(weights):>5}톤 {'':>4} {total_cost:>8,}톤ㆍkm")

    # 창고별 서비스 권역 요약
    print(f"\n > 창고별 서비스 권역:")
    for j in selected:
        served = [demand_cities[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        load = sum(weights[i] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f"{candidates[j]}: {', '.join(served)} (총 {load}톤)")


 풀이 상태 : 최적해
 목적함수값: 13,945 (톤ㆍkm/월)

 > 선택된 창고 (3곳):
- 후보1: 김포
- 후보2: 강동
- 후보3: 의왕

 > 수요 도시별 할당 창고:
수요 도시        수요량배정 창고         거리      기여 비용
--------------------------------------------------
강남         120톤 강동          15km    1,800톤ㆍkm
강북          90톤 김포          25km    2,250톤ㆍkm
인천         150톤 김포          18km    2,700톤ㆍkm
수원         110톤 의왕          15km    1,650톤ㆍkm
성남          80톤 의왕          16km    1,280톤ㆍkm
고양         100톤 김포          20km    2,000톤ㆍkm
부천          95톤 김포          15km    1,425톤ㆍkm
안양          70톤 의왕          12km      840톤ㆍkm
--------------------------------------------------
합계         815톤        13,945톤ㆍkm

 > 창고별 서비스 권역:
김포: 강북, 인천, 고양, 부천 (총 435톤)
강동: 강남 (총 120톤)
의왕: 수원, 성남, 안양 (총 260톤)


1.6

In [18]:
from ortools.sat.python import cp_model

demand_cities = ["강남", "강북", "인천", "수원", "성남", "고양", "부천",
                 "안양"]
candidates = ["김포", "강동", "의왕", "파주", "하남", "안산", "남양주"]

# 각 수요 도시의 월간 수요량 (톤)
weights = [120, 90, 150, 110, 80, 100, 95, 70]

# 거리 행렬 d[i][j]: 수요 도시 i -> 후보 창고 j (km)
# 김포 강동 의왕 파주 하남 안산 남양주
distances = [
    [38, 15, 22, 55, 20, 40, 28], # 강남
    [25, 30, 40, 32, 35, 55, 25], # 강북
    [18, 52, 48, 45, 58, 28, 62], # 인천
    [40, 38, 15, 65, 42, 22, 50], # 수원
    [45, 18, 16, 58, 22, 32, 30], # 성남
    [20, 38, 50, 18, 42, 58, 38], # 고양
    [15, 38, 32, 35, 45, 25, 48], # 부천
    [32, 35, 12, 60, 40, 18, 45], # 안양
]

n = len(demand_cities) # 수요 도시 수
m = len(candidates) # 후보 창고 수
p = 3 # 선택할 창고 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 결정변수
x = [model.NewBoolVar(f"x_{candidates[j]}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{demand_cities[i]}_{candidates[j]}")
     for j in range(m)] for i in range(n)]

# 제약 (1): 정확히 p개 창고 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요 도시는 정확히 1개 창고에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 창고에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 제약 (4): 수요 도시 ~ 배정된 창고 거리 40km 초과 불가
for j in range(m):
    model.Add(2*sum(y[i][j]*weights[i] for i in range(n)) <= sum(weights))

# 목적함수: 가중 거리 합계 최소화
# CP-SAT는 정수형만 처리하므로 실수 거리를 그대로 정수로 사용 (이미 정수 데이터) )
obj_terms = []
for i in range(n):
    for j in range(m):
        obj_terms.append(weights[i] * distances[i][j] * y[i][j])

model.Minimize(sum(obj_terms))

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    print(f"\n 풀이 상태 : {tag}")
    print(f" 목적함수값: {int(solver.ObjectiveValue()):,} (톤ㆍkm/월)")

    # 선택된 창고
    selected = [j for j in range(m) if solver.Value(x[j]) ==1]
    print(f"\n > 선택된 창고 ({p}곳):")
    for j in selected:
        print(f"- 후보{j+1}: {candidates[j]}")

    # 할당 결과
    print(f"\n > 수요 도시별 할당 창고:")
    print(f"{'수요 도시': <8} {'수요량':>7}{'배정 창고':<10}{'거리':>6} {'기여 비용':>10}")
    print(f"{'-'*50}")
    total_cost = 0
    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) ==1)
        cost = weights[i] * distances[i][assigned]
        total_cost += cost
        print(f"{demand_cities[i]:<8} {weights[i]:>5}톤 {candidates[assigned]:<10}"
              f"{distances[i][assigned]:>4}km {cost:>8,}톤ㆍkm")
    print(f"{'-'*50}")
    print(f"{'합계':<8} {sum(weights):>5}톤 {'':>4} {total_cost:>8,}톤ㆍkm")

    # 창고별 서비스 권역 요약
    print(f"\n > 창고별 서비스 권역:")
    for j in selected:
        served = [demand_cities[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        load = sum(weights[i] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f"{candidates[j]}: {', '.join(served)} (총 {load}톤)")


 풀이 상태 : 최적해
 목적함수값: 14,395 (톤ㆍkm/월)

 > 선택된 창고 (3곳):
- 후보1: 김포
- 후보2: 강동
- 후보3: 의왕

 > 수요 도시별 할당 창고:
수요 도시        수요량배정 창고         거리      기여 비용
--------------------------------------------------
강남         120톤 강동          15km    1,800톤ㆍkm
강북          90톤 강동          30km    2,700톤ㆍkm
인천         150톤 김포          18km    2,700톤ㆍkm
수원         110톤 의왕          15km    1,650톤ㆍkm
성남          80톤 의왕          16km    1,280톤ㆍkm
고양         100톤 김포          20km    2,000톤ㆍkm
부천          95톤 김포          15km    1,425톤ㆍkm
안양          70톤 의왕          12km      840톤ㆍkm
--------------------------------------------------
합계         815톤        14,395톤ㆍkm

 > 창고별 서비스 권역:
김포: 인천, 고양, 부천 (총 345톤)
강동: 강남, 강북 (총 210톤)
의왕: 수원, 성남, 안양 (총 260톤)


# 2. P-center 문제

In [1]:
from ortools.sat.python import cp_model

# 데이터
demand_nodes = [
"수원시", "성남시", "안양시", "부천시",
"고양시", "남양주시", "안산시", "광명시",
"평택시", "의정부시"
]
candidates = [
"수원(후보A)", "성남(후보B)", "안양(후보C)", "부천(후보D)",
"고양(후보E)", "하남(후보F)", "안산(후보G)", "의정부(후보H)"
]

# 거리 행렬 d[i][j]: 수요지 i → 후보지 j (분 단위 응급 도달 시간)
# A수원 B성남 C안양 D부천 E고양 F하남 G안산 H의정부
distances = [
[ 5, 28, 22, 38, 48, 30, 25, 55], # 수원시
[28, 5, 18, 40, 42, 15, 38, 42], # 성남시
[22, 18, 5, 30, 40, 25, 20, 50], # 안양시
[38, 40, 30, 5, 22, 42, 28, 38], # 부천시
[48, 42, 40, 22, 5, 35, 45, 20], # 고양시
[30, 15, 25, 42, 35, 5, 42, 30], # 남양주시
[25, 38, 20, 28, 45, 42, 5, 58], # 안산시
[32, 28, 18, 18, 28, 30, 22, 40], # 광명시
[30, 48, 42, 52, 68, 55, 35, 72], # 평택시
[55, 42, 50, 38, 20, 30, 58, 5], # 의정부시
]

n = len(demand_nodes)
m = len(candidates)
p = 3 # 설치할 응급 센터 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 최대 거리값 (상한 계산)
max_dist = max(distances[i][j] for i in range(n) for j in range(m))

# 결정변수
x = [model.NewBoolVar(f"x_{j}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{i}_{j}") for j in range(m)] for i in range(n)]
D = model.NewIntVar(0, max_dist, "D") # 최대 배정 거리 (최소화 대상)

# 제약 (1): 정확히 p개 센터 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요지는 정확히 1개 센터에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 센터에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 제약 (4): 각 수요지의 배정 거리가 D 이하
# Σ_j d_ij * y_ij ≤ D → 선택된 j는 y_ij=1 하나뿐이므로 = d_i(assigned)
for i in range(n):
    model.Add(
        sum(distances[i][j] * y[i][j] for j in range(m)) <= D
    )

# 목적함수: D 최소화 (최악 접근 시간 최소화)
model.Minimize(D)

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

print("=" * 65)
print(" P-Center 문제: 경기도 응급 의료센터 최적 배치 결과")
print("=" * 65)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    opt_D = int(solver.ObjectiveValue())
    print(f"\n 풀이 상태 : {tag}")
    print(f" 최대 도달 시간 (D*): {opt_D}분 ← 이게 최소화된 커버리지 반경")

    selected = [j for j in range(m) if solver.Value(x[j]) == 1]
    print(f"\n ▶ 선택된 응급 센터 ({p}곳):")
    for j in selected:
        print(f" - {candidates[j]}")

    print(f"\n ▶ 수요지별 할당 결과:")
    print(f" {'수요지':<12} {'배정 센터':<16} {'도달 시간':>8} {'최악?':>5}")
    print(f" {'-'*50}")

    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) == 1)
        dist = distances[i][assigned]
        worst = "★ 최악" if dist == opt_D else ""
        print(f" {demand_nodes[i]:<12} {candidates[assigned]:<16} {dist:>5}분 {worst}")
    print(f" {'-'*50}")
    print(f" {'최대 도달 시간':<28} {opt_D:>5}분")

    print(f"\n ▶ 센터별 서비스 권역:")
    for j in selected:
        served = [demand_nodes[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        max_d = max(distances[i][j] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f" {candidates[j]}: {', '.join(served)} (최대 {max_d}분)")
else:
    print(" 풀이 실패: 해를 찾지 못했습니다.")

print(f"\n 풀이 시간: {solver.WallTime():.4f}초")
print("=" * 65)

 P-Center 문제: 경기도 응급 의료센터 최적 배치 결과

 풀이 상태 : 최적해
 최대 도달 시간 (D*): 30분 ← 이게 최소화된 커버리지 반경

 ▶ 선택된 응급 센터 (3곳):
 - 수원(후보A)
 - 안양(후보C)
 - 고양(후보E)

 ▶ 수요지별 할당 결과:
 수요지          배정 센터               도달 시간   최악?
 --------------------------------------------------
 수원시          수원(후보A)              5분 
 성남시          수원(후보A)             28분 
 안양시          수원(후보A)             22분 
 부천시          안양(후보C)             30분 ★ 최악
 고양시          고양(후보E)              5분 
 남양주시         수원(후보A)             30분 ★ 최악
 안산시          수원(후보A)             25분 
 광명시          안양(후보C)             18분 
 평택시          수원(후보A)             30분 ★ 최악
 의정부시         고양(후보E)             20분 
 --------------------------------------------------
 최대 도달 시간                        30분

 ▶ 센터별 서비스 권역:
 수원(후보A): 수원시, 성남시, 안양시, 남양주시, 안산시, 평택시 (최대 30분)
 안양(후보C): 부천시, 광명시 (최대 30분)
 고양(후보E): 고양시, 의정부시 (최대 20분)

 풀이 시간: 0.1037초


### 2.1 

In [3]:

from ortools.sat.python import cp_model

# 데이터
demand_nodes = [
"수원시", "성남시", "안양시", "부천시",
"고양시", "남양주시", "안산시", "광명시",
"평택시", "의정부시"
]
candidates = [
"수원(후보A)", "성남(후보B)", "안양(후보C)", "부천(후보D)",
"고양(후보E)", "하남(후보F)", "안산(후보G)", "의정부(후보H)"
]

# 거리 행렬 d[i][j]: 수요지 i → 후보지 j (분 단위 응급 도달 시간)
# A수원 B성남 C안양 D부천 E고양 F하남 G안산 H의정부
distances = [
[ 5, 28, 22, 38, 48, 30, 25, 55], # 수원시
[28, 5, 18, 40, 42, 15, 38, 42], # 성남시
[22, 18, 5, 30, 40, 25, 20, 50], # 안양시
[38, 40, 30, 5, 22, 42, 28, 38], # 부천시
[48, 42, 40, 22, 5, 35, 45, 20], # 고양시
[30, 15, 25, 42, 35, 5, 42, 30], # 남양주시
[25, 38, 20, 28, 45, 42, 5, 58], # 안산시
[32, 28, 18, 18, 28, 30, 22, 40], # 광명시
[30, 48, 42, 52, 68, 55, 35, 72], # 평택시
[55, 42, 50, 38, 20, 30, 58, 5], # 의정부시
]

n = len(demand_nodes)
m = len(candidates)
p = 5 # 설치할 응급 센터 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 최대 거리값 (상한 계산)
max_dist = max(distances[i][j] for i in range(n) for j in range(m))

# 결정변수
x = [model.NewBoolVar(f"x_{j}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{i}_{j}") for j in range(m)] for i in range(n)]
D = model.NewIntVar(0, max_dist, "D") # 최대 배정 거리 (최소화 대상)

# 제약 (1): 정확히 p개 센터 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요지는 정확히 1개 센터에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 추가 제약 : 선택된 센터가 반드시 수요지를 가져야 한다
for j in range(m):
    model.Add(sum(y[i][j] for i in range(n)) >= x[j])

# 제약 (3): 선택된 센터에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 제약 (4): 각 수요지의 배정 거리가 D 이하
# Σ_j d_ij * y_ij ≤ D → 선택된 j는 y_ij=1 하나뿐이므로 = d_i(assigned)
for i in range(n):
    model.Add(
        sum(distances[i][j] * y[i][j] for j in range(m)) <= D
    )

# 목적함수: D 최소화 (최악 접근 시간 최소화)
model.Minimize(D)

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

print("=" * 65)
print(" P-Center 문제: 경기도 응급 의료센터 최적 배치 결과")
print("=" * 65)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    opt_D = int(solver.ObjectiveValue())
    print(f"\n 풀이 상태 : {tag}")
    print(f" 최대 도달 시간 (D*): {opt_D}분 ← 이게 최소화된 커버리지 반경")

    selected = [j for j in range(m) if solver.Value(x[j]) == 1]
    print(f"\n ▶ 선택된 응급 센터 ({p}곳):")
    for j in selected:
        print(f" - {candidates[j]}")

    print(f"\n ▶ 수요지별 할당 결과:")
    print(f" {'수요지':<12} {'배정 센터':<16} {'도달 시간':>8} {'최악?':>5}")
    print(f" {'-'*50}")

    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) == 1)
        dist = distances[i][assigned]
        worst = "★ 최악" if dist == opt_D else ""
        print(f" {demand_nodes[i]:<12} {candidates[assigned]:<16} {dist:>5}분 {worst}")
    print(f" {'-'*50}")
    print(f" {'최대 도달 시간':<28} {opt_D:>5}분")

    print(f"\n ▶ 센터별 서비스 권역:")
    for j in selected:
        served = [demand_nodes[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        max_d = max(distances[i][j] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f" {candidates[j]}: {', '.join(served)} (최대 {max_d}분)")
else:
    print(" 풀이 실패: 해를 찾지 못했습니다.")

print(f"\n 풀이 시간: {solver.WallTime():.4f}초")
print("=" * 65)

 P-Center 문제: 경기도 응급 의료센터 최적 배치 결과

 풀이 상태 : 최적해
 최대 도달 시간 (D*): 30분 ← 이게 최소화된 커버리지 반경

 ▶ 선택된 응급 센터 (5곳):
 - 수원(후보A)
 - 성남(후보B)
 - 고양(후보E)
 - 안산(후보G)
 - 의정부(후보H)

 ▶ 수요지별 할당 결과:
 수요지          배정 센터               도달 시간   최악?
 --------------------------------------------------
 수원시          안산(후보G)             25분 
 성남시          성남(후보B)              5분 
 안양시          수원(후보A)             22분 
 부천시          고양(후보E)             22분 
 고양시          의정부(후보H)            20분 
 남양주시         의정부(후보H)            30분 ★ 최악
 안산시          수원(후보A)             25분 
 광명시          성남(후보B)             28분 
 평택시          수원(후보A)             30분 ★ 최악
 의정부시         의정부(후보H)             5분 
 --------------------------------------------------
 최대 도달 시간                        30분

 ▶ 센터별 서비스 권역:
 수원(후보A): 안양시, 안산시, 평택시 (최대 30분)
 성남(후보B): 성남시, 광명시 (최대 28분)
 고양(후보E): 부천시 (최대 22분)
 안산(후보G): 수원시 (최대 25분)
 의정부(후보H): 고양시, 남양주시, 의정부시 (최대 30분)

 풀이 시간: 0.0603초


D*은 30분에서 변동 x

### 2.2

In [8]:

from ortools.sat.python import cp_model

# 데이터
demand_nodes = [
"수원시", "성남시", "안양시", "부천시",
"고양시", "남양주시", "안산시", "광명시",
"평택시", "의정부시"
]
candidates = [
"수원(후보A)", "성남(후보B)", "안양(후보C)", "부천(후보D)",
"고양(후보E)", "하남(후보F)", "안산(후보G)", "의정부(후보H)"
]

# 거리 행렬 d[i][j]: 수요지 i → 후보지 j (분 단위 응급 도달 시간)
# A수원 B성남 C안양 D부천 E고양 F하남 G안산 H의정부
distances = [
[ 5, 28, 22, 38, 48, 30, 25, 55], # 수원시
[28, 5, 18, 40, 42, 15, 38, 42], # 성남시
[22, 18, 5, 30, 40, 25, 20, 50], # 안양시
[38, 40, 30, 5, 22, 42, 28, 38], # 부천시
[48, 42, 40, 22, 5, 35, 45, 20], # 고양시
[30, 15, 25, 42, 35, 5, 42, 30], # 남양주시
[25, 38, 20, 28, 45, 42, 5, 58], # 안산시
[32, 28, 18, 18, 28, 30, 22, 40], # 광명시
[30, 48, 42, 52, 68, 55, 35, 72], # 평택시
[55, 42, 50, 38, 20, 30, 58, 5], # 의정부시
]

n = len(demand_nodes)
m = len(candidates)
p = 3 # 설치할 응급 센터 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 최대 거리값 (상한 계산)
max_dist = max(distances[i][j] for i in range(n) for j in range(m))

# 결정변수
x = [model.NewBoolVar(f"x_{j}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{i}_{j}") for j in range(m)] for i in range(n)]
D = model.NewIntVar(0, max_dist, "D") # 최대 배정 거리 (최소화 대상)

# 제약 (1): 정확히 p개 센터 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요지는 정확히 1개 센터에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 추가 제약2 : 의정부 센터(후보 H, 인덱스 7)는 반드시 설치되어야 한다.
model.Add(x[7] == 1)

# 제약 (3): 선택된 센터에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 제약 (4): 각 수요지의 배정 거리가 D 이하
# Σ_j d_ij * y_ij ≤ D → 선택된 j는 y_ij=1 하나뿐이므로 = d_i(assigned)
for i in range(n):
    model.Add(
        sum(distances[i][j] * y[i][j] for j in range(m)) <= D
    )

# 목적함수: D 최소화 (최악 접근 시간 최소화)
model.Minimize(D)

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

print("=" * 65)
print(" P-Center 문제: 경기도 응급 의료센터 최적 배치 결과")
print("=" * 65)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    opt_D = int(solver.ObjectiveValue())
    print(f"\n 풀이 상태 : {tag}")
    print(f" 최대 도달 시간 (D*): {opt_D}분 ← 이게 최소화된 커버리지 반경")

    selected = [j for j in range(m) if solver.Value(x[j]) == 1]
    print(f"\n ▶ 선택된 응급 센터 ({p}곳):")
    for j in selected:
        print(f" - {candidates[j]}")

    print(f"\n ▶ 수요지별 할당 결과:")
    print(f" {'수요지':<12} {'배정 센터':<16} {'도달 시간':>8} {'최악?':>5}")
    print(f" {'-'*50}")

    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) == 1)
        dist = distances[i][assigned]
        worst = "★ 최악" if dist == opt_D else ""
        print(f" {demand_nodes[i]:<12} {candidates[assigned]:<16} {dist:>5}분 {worst}")
    print(f" {'-'*50}")
    print(f" {'최대 도달 시간':<28} {opt_D:>5}분")

    print(f"\n ▶ 센터별 서비스 권역:")
    for j in selected:
        served = [demand_nodes[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        max_d = max(distances[i][j] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f" {candidates[j]}: {', '.join(served)} (최대 {max_d}분)")
else:
    print(" 풀이 실패: 해를 찾지 못했습니다.")

print(f"\n 풀이 시간: {solver.WallTime():.4f}초")
print("=" * 65)

 P-Center 문제: 경기도 응급 의료센터 최적 배치 결과

 풀이 상태 : 최적해
 최대 도달 시간 (D*): 30분 ← 이게 최소화된 커버리지 반경

 ▶ 선택된 응급 센터 (3곳):
 - 수원(후보A)
 - 안양(후보C)
 - 의정부(후보H)

 ▶ 수요지별 할당 결과:
 수요지          배정 센터               도달 시간   최악?
 --------------------------------------------------
 수원시          안양(후보C)             22분 
 성남시          안양(후보C)             18분 
 안양시          수원(후보A)             22분 
 부천시          안양(후보C)             30분 ★ 최악
 고양시          의정부(후보H)            20분 
 남양주시         의정부(후보H)            30분 ★ 최악
 안산시          수원(후보A)             25분 
 광명시          안양(후보C)             18분 
 평택시          수원(후보A)             30분 ★ 최악
 의정부시         의정부(후보H)             5분 
 --------------------------------------------------
 최대 도달 시간                        30분

 ▶ 센터별 서비스 권역:
 수원(후보A): 안양시, 안산시, 평택시 (최대 30분)
 안양(후보C): 수원시, 성남시, 부천시, 광명시 (최대 30분)
 의정부(후보H): 고양시, 남양주시, 의정부시 (최대 30분)

 풀이 시간: 0.0306초


### 2.3

In [9]:

from ortools.sat.python import cp_model

# 데이터
demand_nodes = [
"수원시", "성남시", "안양시", "부천시",
"고양시", "남양주시", "안산시", "광명시",
"평택시", "의정부시"
]
candidates = [
"수원(후보A)", "성남(후보B)", "안양(후보C)", "부천(후보D)",
"고양(후보E)", "하남(후보F)", "안산(후보G)", "의정부(후보H)"
]

# 거리 행렬 d[i][j]: 수요지 i → 후보지 j (분 단위 응급 도달 시간)
# A수원 B성남 C안양 D부천 E고양 F하남 G안산 H의정부
distances = [
[ 5, 28, 22, 38, 48, 30, 25, 55], # 수원시
[28, 5, 18, 40, 42, 15, 38, 42], # 성남시
[22, 18, 5, 30, 40, 25, 20, 50], # 안양시
[38, 40, 30, 5, 22, 42, 28, 38], # 부천시
[48, 42, 40, 22, 5, 35, 45, 20], # 고양시
[30, 15, 25, 42, 35, 5, 42, 30], # 남양주시
[25, 38, 20, 28, 45, 42, 5, 58], # 안산시
[32, 28, 18, 18, 28, 30, 22, 40], # 광명시
[30, 48, 42, 52, 68, 55, 35, 72], # 평택시
[55, 42, 50, 38, 20, 30, 58, 5], # 의정부시
]

n = len(demand_nodes)
m = len(candidates)
p = 3 # 설치할 응급 센터 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 최대 거리값 (상한 계산)
max_dist = max(distances[i][j] for i in range(n) for j in range(m))

# 결정변수
x = [model.NewBoolVar(f"x_{j}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{i}_{j}") for j in range(m)] for i in range(n)]
D = model.NewIntVar(0, max_dist, "D") # 최대 배정 거리 (최소화 대상)

# 제약 (1): 정확히 p개 센터 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요지는 정확히 1개 센터에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 추가 제약 : 안산에는 설치되면 안된다.
model.Add(x[6] == 0)

# 제약 (3): 선택된 센터에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 제약 (4): 각 수요지의 배정 거리가 D 이하
# Σ_j d_ij * y_ij ≤ D → 선택된 j는 y_ij=1 하나뿐이므로 = d_i(assigned)
for i in range(n):
    model.Add(
        sum(distances[i][j] * y[i][j] for j in range(m)) <= D
    )

# 목적함수: D 최소화 (최악 접근 시간 최소화)
model.Minimize(D)

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

print("=" * 65)
print(" P-Center 문제: 경기도 응급 의료센터 최적 배치 결과")
print("=" * 65)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    opt_D = int(solver.ObjectiveValue())
    print(f"\n 풀이 상태 : {tag}")
    print(f" 최대 도달 시간 (D*): {opt_D}분 ← 이게 최소화된 커버리지 반경")

    selected = [j for j in range(m) if solver.Value(x[j]) == 1]
    print(f"\n ▶ 선택된 응급 센터 ({p}곳):")
    for j in selected:
        print(f" - {candidates[j]}")

    print(f"\n ▶ 수요지별 할당 결과:")
    print(f" {'수요지':<12} {'배정 센터':<16} {'도달 시간':>8} {'최악?':>5}")
    print(f" {'-'*50}")

    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) == 1)
        dist = distances[i][assigned]
        worst = "★ 최악" if dist == opt_D else ""
        print(f" {demand_nodes[i]:<12} {candidates[assigned]:<16} {dist:>5}분 {worst}")
    print(f" {'-'*50}")
    print(f" {'최대 도달 시간':<28} {opt_D:>5}분")

    print(f"\n ▶ 센터별 서비스 권역:")
    for j in selected:
        served = [demand_nodes[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        max_d = max(distances[i][j] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f" {candidates[j]}: {', '.join(served)} (최대 {max_d}분)")
else:
    print(" 풀이 실패: 해를 찾지 못했습니다.")

print(f"\n 풀이 시간: {solver.WallTime():.4f}초")
print("=" * 65)

 P-Center 문제: 경기도 응급 의료센터 최적 배치 결과

 풀이 상태 : 최적해
 최대 도달 시간 (D*): 30분 ← 이게 최소화된 커버리지 반경

 ▶ 선택된 응급 센터 (3곳):
 - 수원(후보A)
 - 성남(후보B)
 - 고양(후보E)

 ▶ 수요지별 할당 결과:
 수요지          배정 센터               도달 시간   최악?
 --------------------------------------------------
 수원시          수원(후보A)              5분 
 성남시          수원(후보A)             28분 
 안양시          수원(후보A)             22분 
 부천시          고양(후보E)             22분 
 고양시          고양(후보E)              5분 
 남양주시         수원(후보A)             30분 ★ 최악
 안산시          수원(후보A)             25분 
 광명시          성남(후보B)             28분 
 평택시          수원(후보A)             30분 ★ 최악
 의정부시         고양(후보E)             20분 
 --------------------------------------------------
 최대 도달 시간                        30분

 ▶ 센터별 서비스 권역:
 수원(후보A): 수원시, 성남시, 안양시, 남양주시, 안산시, 평택시 (최대 30분)
 성남(후보B): 광명시 (최대 28분)
 고양(후보E): 부천시, 고양시, 의정부시 (최대 22분)

 풀이 시간: 0.0177초


### 2.4

In [11]:

from ortools.sat.python import cp_model

# 데이터
demand_nodes = [
"수원시", "성남시", "안양시", "부천시",
"고양시", "남양주시", "안산시", "광명시",
"평택시", "의정부시"
]
candidates = [
"수원(후보A)", "성남(후보B)", "안양(후보C)", "부천(후보D)",
"고양(후보E)", "하남(후보F)", "안산(후보G)", "의정부(후보H)"
]

# 거리 행렬 d[i][j]: 수요지 i → 후보지 j (분 단위 응급 도달 시간)
# A수원 B성남 C안양 D부천 E고양 F하남 G안산 H의정부
distances = [
[ 5, 28, 22, 38, 48, 30, 25, 55], # 수원시
[28, 5, 18, 40, 42, 15, 38, 42], # 성남시
[22, 18, 5, 30, 40, 25, 20, 50], # 안양시
[38, 40, 30, 5, 22, 42, 28, 38], # 부천시
[48, 42, 40, 22, 5, 35, 45, 20], # 고양시
[30, 15, 25, 42, 35, 5, 42, 30], # 남양주시
[25, 38, 20, 28, 45, 42, 5, 58], # 안산시
[32, 28, 18, 18, 28, 30, 22, 40], # 광명시
[30, 48, 42, 52, 68, 55, 35, 72], # 평택시
[55, 42, 50, 38, 20, 30, 58, 5], # 의정부시
]

n = len(demand_nodes)
m = len(candidates)
# p = 3 # 설치할 응급 센터 수
D=25

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 최대 거리값 (상한 계산)
max_dist = max(distances[i][j] for i in range(n) for j in range(m))

# 결정변수
x = [model.NewBoolVar(f"x_{j}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{i}_{j}") for j in range(m)] for i in range(n)]
p = model.NewIntVar(0, m, "p")
# D = model.NewIntVar(0, max_dist, "D") # 최대 배정 거리 (최소화 대상)

# 제약 (1): 정확히 p개 센터 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요지는 정확히 1개 센터에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 센터에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 제약 (4): 각 수요지의 배정 거리가 D 이하
# Σ_j d_ij * y_ij ≤ D → 선택된 j는 y_ij=1 하나뿐이므로 = d_i(assigned)
for i in range(n):
    model.Add(
        sum(distances[i][j] * y[i][j] for j in range(m)) <= D
    )

# 목적함수: D 최소화 (최악 접근 시간 최소화)
model.Minimize(D)

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

print("=" * 65)
print(" P-Center 문제: 경기도 응급 의료센터 최적 배치 결과")
print("=" * 65)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    opt_D = int(solver.ObjectiveValue())
    print(f"\n 풀이 상태 : {tag}")
    print(f" 최대 도달 시간 (D*): {opt_D}분 ← 이게 최소화된 커버리지 반경")

    selected = [j for j in range(m) if solver.Value(x[j]) == 1]
    print(f"\n ▶ 선택된 응급 센터 ({p}곳):")
    for j in selected:
        print(f" - {candidates[j]}")

    print(f"\n ▶ 수요지별 할당 결과:")
    print(f" {'수요지':<12} {'배정 센터':<16} {'도달 시간':>8} {'최악?':>5}")
    print(f" {'-'*50}")

    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) == 1)
        dist = distances[i][assigned]
        worst = "★ 최악" if dist == opt_D else ""
        print(f" {demand_nodes[i]:<12} {candidates[assigned]:<16} {dist:>5}분 {worst}")
    print(f" {'-'*50}")
    print(f" {'최대 도달 시간':<28} {opt_D:>5}분")

    print(f"\n ▶ 센터별 서비스 권역:")
    for j in selected:
        served = [demand_nodes[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        max_d = max(distances[i][j] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f" {candidates[j]}: {', '.join(served)} (최대 {max_d}분)")
else:
    print(" 풀이 실패: 해를 찾지 못했습니다.")

print(f"\n 풀이 시간: {solver.WallTime():.4f}초")
print("=" * 65)

 P-Center 문제: 경기도 응급 의료센터 최적 배치 결과
 풀이 실패: 해를 찾지 못했습니다.

 풀이 시간: 0.0024초


25분 초과할 경우 p=3 설정에서 기준 충족 불가하며  
현재 주어진 n개의 센터내에서는 어떠한 경우에도 불가하다.

### 2.5

In [1]:

from ortools.sat.python import cp_model

# 데이터
demand_nodes = [
"수원시", "성남시", "안양시", "부천시",
"고양시", "남양주시", "안산시", "광명시",
"평택시", "의정부시"
]
candidates = [
"수원(후보A)", "성남(후보B)", "안양(후보C)", "부천(후보D)",
"고양(후보E)", "하남(후보F)", "안산(후보G)", "의정부(후보H)"
]

# 거리 행렬 d[i][j]: 수요지 i → 후보지 j (분 단위 응급 도달 시간)
# A수원 B성남 C안양 D부천 E고양 F하남 G안산 H의정부
distances = [
[ 5, 28, 22, 38, 48, 30, 25, 55], # 수원시
[28, 5, 18, 40, 42, 15, 38, 42], # 성남시
[22, 18, 5, 30, 40, 25, 20, 50], # 안양시
[38, 40, 30, 5, 22, 42, 28, 38], # 부천시
[48, 42, 40, 22, 5, 35, 45, 20], # 고양시
[30, 15, 25, 42, 35, 5, 42, 30], # 남양주시
[25, 38, 20, 28, 45, 42, 5, 58], # 안산시
[32, 28, 18, 18, 28, 30, 22, 40], # 광명시
[30, 48, 42, 52, 68, 55, 35, 72], # 평택시
[55, 42, 50, 38, 20, 30, 58, 5], # 의정부시
]

n = len(demand_nodes)
m = len(candidates)
p = 3 # 설치할 응급 센터 수

model = cp_model.CpModel()
solver = cp_model.CpSolver()

# 최대 거리값 (상한 계산)
max_dist = max(distances[i][j] for i in range(n) for j in range(m))

# 결정변수
x = [model.NewBoolVar(f"x_{j}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{i}_{j}") for j in range(m)] for i in range(n)]
D = model.NewIntVar(0, max_dist, "D") # 최대 배정 거리 (최소화 대상)

# 제약 (1): 정확히 p개 센터 선택
model.Add(sum(x) == p)

# 제약 (2): 각 수요지는 정확히 1개 센터에 할당
for i in range(n):
    model.Add(sum(y[i]) == 1)

# 제약 (3): 선택된 센터에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# 제약 (4): 각 수요지의 배정 거리가 D 이하
# Σ_j d_ij * y_ij ≤ D → 선택된 j는 y_ij=1 하나뿐이므로 = d_i(assigned)
for i in range(n):
    model.Add(
        sum(distances[i][j] * y[i][j] for j in range(m)) <= D
    )

# 추가제약: 고양 또는 의정부에 하나, 수원 또는 안산에 하나
model.Add(x[4]+x[7] == 1)
model.Add(x[0]+x[6] == 1)

# 목적함수: D 최소화 (최악 접근 시간 최소화)
model.Minimize(D)

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

print("=" * 65)
print(" P-Center 문제: 경기도 응급 의료센터 최적 배치 결과")
print("=" * 65)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    opt_D = int(solver.ObjectiveValue())
    print(f"\n 풀이 상태 : {tag}")
    print(f" 최대 도달 시간 (D*): {opt_D}분 ← 이게 최소화된 커버리지 반경")

    selected = [j for j in range(m) if solver.Value(x[j]) == 1]
    print(f"\n ▶ 선택된 응급 센터 ({p}곳):")
    for j in selected:
        print(f" - {candidates[j]}")

    print(f"\n ▶ 수요지별 할당 결과:")
    print(f" {'수요지':<12} {'배정 센터':<16} {'도달 시간':>8} {'최악?':>5}")
    print(f" {'-'*50}")

    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) == 1)
        dist = distances[i][assigned]
        worst = "★ 최악" if dist == opt_D else ""
        print(f" {demand_nodes[i]:<12} {candidates[assigned]:<16} {dist:>5}분 {worst}")
    print(f" {'-'*50}")
    print(f" {'최대 도달 시간':<28} {opt_D:>5}분")

    print(f"\n ▶ 센터별 서비스 권역:")
    for j in selected:
        served = [demand_nodes[i] for i in range(n) if solver.Value(y[i][j]) == 1]
        max_d = max(distances[i][j] for i in range(n) if solver.Value(y[i][j]) == 1)
        print(f" {candidates[j]}: {', '.join(served)} (최대 {max_d}분)")
else:
    print(" 풀이 실패: 해를 찾지 못했습니다.")

print(f"\n 풀이 시간: {solver.WallTime():.4f}초")
print("=" * 65)

 P-Center 문제: 경기도 응급 의료센터 최적 배치 결과

 풀이 상태 : 최적해
 최대 도달 시간 (D*): 30분 ← 이게 최소화된 커버리지 반경

 ▶ 선택된 응급 센터 (3곳):
 - 수원(후보A)
 - 부천(후보D)
 - 고양(후보E)

 ▶ 수요지별 할당 결과:
 수요지          배정 센터               도달 시간   최악?
 --------------------------------------------------
 수원시          수원(후보A)              5분 
 성남시          수원(후보A)             28분 
 안양시          부천(후보D)             30분 ★ 최악
 부천시          부천(후보D)              5분 
 고양시          부천(후보D)             22분 
 남양주시         수원(후보A)             30분 ★ 최악
 안산시          부천(후보D)             28분 
 광명시          부천(후보D)             18분 
 평택시          수원(후보A)             30분 ★ 최악
 의정부시         고양(후보E)             20분 
 --------------------------------------------------
 최대 도달 시간                        30분

 ▶ 센터별 서비스 권역:
 수원(후보A): 수원시, 성남시, 남양주시, 평택시 (최대 30분)
 부천(후보D): 안양시, 부천시, 고양시, 안산시, 광명시 (최대 30분)
 고양(후보E): 의정부시 (최대 20분)

 풀이 시간: 0.0771초


### 2.6

In [22]:
from ortools.sat.python import cp_model

# 데이터
demand_nodes = [
    "수원시", "성남시", "안양시", "부천시",
    "고양시", "남양주시", "안산시", "광명시",
    "평택시", "의정부시"
]
candidates = [
    "수원(후보A)", "성남(후보B)", "안양(후보C)", "부천(후보D)",
    "고양(후보E)", "하남(후보F)", "안산(후보G)", "의정부(후보H)"
]

# 거리 행렬
distances = [
    [ 5, 28, 22, 38, 48, 30, 25, 55],  # 수원시
    [28, 5, 18, 40, 42, 15, 38, 42],   # 성남시
    [22, 18, 5, 30, 40, 25, 20, 50],   # 안양시
    [38, 40, 30, 5, 22, 42, 28, 38],   # 부천시
    [48, 42, 40, 22, 5, 35, 45, 20],   # 고양시
    [30, 15, 25, 42, 35, 5, 42, 30],   # 남양주시
    [25, 38, 20, 28, 45, 42, 5, 58],   # 안산시
    [32, 28, 18, 18, 28, 30, 22, 40],  # 광명시
    [30, 48, 42, 52, 68, 55, 35, 72],  # 평택시
    [55, 42, 50, 38, 20, 30, 58, 5],   # 의정부시
]

n = len(demand_nodes)
m = len(candidates)
p = 3
limit_D = 30

model = cp_model.CpModel()
solver = cp_model.CpSolver()

max_dist = max(distances[i][j] for i in range(n) for j in range(m))
max_total = sum(max(row) for row in distances)

# 결정변수
x = [model.NewBoolVar(f"x_{j}") for j in range(m)]
y = [[model.NewBoolVar(f"y_{i}_{j}") for j in range(m)] for i in range(n)]
D = model.NewIntVar(0, max_dist, "D")
total_distance = model.NewIntVar(0, max_total, "total_distance")

# (1) 정확히 5개 센터 설치
model.Add(sum(x) == p)

# (2) 각 수요지는 정확히 1개 센터에 할당
for i in range(n):
    model.Add(sum(y[i][j] for j in range(m)) == 1)

# (3) 선택된 센터에만 할당 가능
for i in range(n):
    for j in range(m):
        model.Add(y[i][j] <= x[j])

# (4) 각 수요지의 배정 거리는 D 이하
for i in range(n):
    model.Add(sum(distances[i][j] * y[i][j] for j in range(m)) <= D)

# (5) 최악 도달 시간 30분 이하 보장
model.Add(D <= limit_D)

# 전체 도달 시간 합계
model.Add(
    total_distance ==
    sum(distances[i][j] * y[i][j] for i in range(n) for j in range(m))
)

# 목적함수: 전체 도달 시간 합계 최소화
model.Minimize(total_distance)

solver.parameters.max_time_in_seconds = 60.0
status = solver.Solve(model)

print("=" * 70)
print(" 응급센터 배치: D<=30 보장 + 전체 도달시간 합 최소화")
print("=" * 70)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tag = "최적해" if status == cp_model.OPTIMAL else "실행가능해"
    opt_D = solver.Value(D)
    opt_total = solver.Value(total_distance)
    avg_time = opt_total / n

    print(f"\n풀이 상태 : {tag}")
    print(f"최악 도달 시간 D* : {opt_D}분")
    print(f"전체 도달 시간 합 : {opt_total}분")
    print(f"평균 도달 시간   : {avg_time:.2f}분")

    selected = [j for j in range(m) if solver.Value(x[j]) == 1]
    print(f"\n▶ 선택된 응급센터 ({p}곳)")
    for j in selected:
        print(f" - {candidates[j]}")

    print(f"\n▶ 수요지별 할당 결과")
    print(f"{'수요지':<10} {'배정 센터':<14} {'도달 시간':>8} {'최악?':>6}")
    print("-" * 50)

    for i in range(n):
        assigned = next(j for j in range(m) if solver.Value(y[i][j]) == 1)
        dist = distances[i][assigned]
        mark = "★" if dist == opt_D else ""
        print(f"{demand_nodes[i]:<10} {candidates[assigned]:<14} {dist:>5}분 {mark:>4}")

    print("-" * 50)
    print(f"{'최악 도달 시간':<25} {opt_D:>5}분")
    print(f"{'전체 도달 시간 합':<25} {opt_total:>5}분")
    print(f"{'평균 도달 시간':<25} {avg_time:>5.2f}분")

    print(f"\n▶ 센터별 서비스 권역")
    for j in selected:
        served_idx = [i for i in range(n) if solver.Value(y[i][j]) == 1]
        served = [demand_nodes[i] for i in served_idx]

        if served_idx:
            max_d = max(distances[i][j] for i in served_idx)
            print(f"{candidates[j]}: {', '.join(served)} (최대 {max_d}분)")
        else:
            print(f"{candidates[j]}: 배정된 수요지 없음")

else:
    print("해를 찾지 못했습니다.")

print(f"\n풀이 시간: {solver.WallTime():.4f}초")
print("=" * 70)

 응급센터 배치: D<=30 보장 + 전체 도달시간 합 최소화

풀이 상태 : 최적해
최악 도달 시간 D* : 30분
전체 도달 시간 합 : 168분
평균 도달 시간   : 16.80분

▶ 선택된 응급센터 (3곳)
 - 수원(후보A)
 - 안양(후보C)
 - 고양(후보E)

▶ 수요지별 할당 결과
수요지        배정 센터             도달 시간    최악?
--------------------------------------------------
수원시        수원(후보A)            5분     
성남시        안양(후보C)           18분     
안양시        안양(후보C)            5분     
부천시        고양(후보E)           22분     
고양시        고양(후보E)            5분     
남양주시       안양(후보C)           25분     
안산시        안양(후보C)           20분     
광명시        안양(후보C)           18분     
평택시        수원(후보A)           30분    ★
의정부시       고양(후보E)           20분     
--------------------------------------------------
최악 도달 시간                     30분
전체 도달 시간 합                  168분
평균 도달 시간                  16.80분

▶ 센터별 서비스 권역
수원(후보A): 수원시, 평택시 (최대 30분)
안양(후보C): 성남시, 안양시, 남양주시, 안산시, 광명시 (최대 25분)
고양(후보E): 부천시, 고양시, 의정부시 (최대 22분)

풀이 시간: 0.0307초
